In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchaudio
import torchaudio.transforms as T
import matplotlib.pyplot as plt
import numpy as np
import os
import glob
from tqdm import tqdm


Config

In [ ]:
CONFIG = {
    "sample_rate": 24000,      # Sample Rate ที่นิยมใช้ใน RVC
    "max_length": 24000 * 3,   # ตัดเสียงให้เหลือไฟล์ละ 3 วินาที (เพื่อเทรนง่าย)
    "n_mels": 80,
    "batch_size": 16,
    "epochs": 50,
    "save_interval": 10,       # Save Model ทุกๆ 10 Epochs
    "device": "cuda" if torch.cuda.is_available() else "cpu"
}


In [ ]:
class AnimeVoiceDataset(torch.utils.data.Dataset):
    def __init__(self, root_dir, sample_rate=24000, max_len=72000):
        self.root_dir = root_dir
        self.sample_rate = sample_rate
        self.max_len = max_len
        self.files = glob.glob(os.path.join(root_dir, "*.wav"))
        
        if len(self.files) == 0:
            print(f"ไม่พบไฟล์ .wav")

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        file_path = self.files[idx]
        waveform, sr = torchaudio.load(file_path)

        # 1. Resample ถ้า Sample rate ไม่ตรง
        if sr != self.sample_rate:
            resampler = T.Resample(sr, self.sample_rate)
            waveform = resampler(waveform)

        # 2. Convert to Mono (ถ้าเป็น Stereo)
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)

        # 3. Pad or Trim (ทำให้ความยาวเท่ากันทุกไฟล์ เพื่อเข้า Batch)
        if waveform.shape[1] > self.max_len:
            start = torch.randint(0, waveform.shape[1] - self.max_len, (1,))
            waveform = waveform[:, start:start+self.max_len]
        elif waveform.shape[1] < self.max_len:
            padding = self.max_len - waveform.shape[1]
            waveform = torch.nn.functional.pad(waveform, (0, padding))

        return waveform


Class Metric Performance Model

In [ ]:
class AudioMetrics:
    def __init__(self, sample_rate):
        self.mel_transform = T.MelSpectrogram(
            sample_rate=sample_rate, n_fft=1024, win_length=1024, hop_length=256, n_mels=80
        ).to(CONFIG['device'])
        self.l1_loss = nn.L1Loss()

    def calc_loss(self, gen_wave, target_wave):
        # 1. Mel-Spectrogram Loss (สำคัญสุดสำหรับคุณภาพเสียง)
        gen_mel = self.mel_transform(gen_wave)
        tgt_mel = self.mel_transform(target_wave)
        mel_loss = self.l1_loss(gen_mel, tgt_mel)
        
        # 2. Waveform Loss (ความเหมือนของคลื่นเสียงดิบ)
        wave_loss = self.l1_loss(gen_wave, target_wave)
        
        # Total Loss (ถ่วงน้ำหนัก Mel เยอะหน่อยเพราะหูคนไวต่อความถี่)
        total_loss = mel_loss + (wave_loss * 0.5)
        
        return total_loss, mel_loss, wave_loss, gen_mel, tgt_mel


plot Graph

In [ ]:
def plot_metrics(loss_history, gen, tgt, gen_mel, tgt_mel, epoch, save_dir="logs_plots"):
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)
        
    plt.figure(figsize=(12, 8))
    
    # 1. Loss Curve
    plt.subplot(2, 2, 1)
    plt.plot(loss_history, label='Mel Loss')
    plt.title(f"Training Loss (Epoch {epoch})")
    plt.grid(alpha=0.3)

    # 2. Waveform
    plt.subplot(2, 2, 2)
    plt.plot(tgt[0].cpu().detach().numpy().flatten()[:1000], alpha=0.5, label='Target')
    plt.plot(gen[0].cpu().detach().numpy().flatten()[:1000], alpha=0.7, label='Generated')
    plt.title("Waveform Comparison")
    plt.legend()

    # 3. Spectrograms
    plt.subplot(2, 2, 3)
    plt.imshow(np.log(gen_mel[0].cpu().detach().numpy() + 1e-9), aspect='auto', origin='lower')
    plt.title("Generated Spec")
    
    plt.subplot(2, 2, 4)
    plt.imshow(np.log(tgt_mel[0].cpu().detach().numpy() + 1e-9), aspect='auto', origin='lower')
    plt.title("Target Spec")

    plt.tight_layout()
    plt.close()


make Neural  Model ARCHITECTURE

In [ ]:
class RVC_AnimeModel(nn.Module):
    def __init__(self):
        super().__init__()
        # ใช้ Conv1d ง่ายๆ จำลองการทำงาน (ของจริงซับซ้อนกว่านี้มาก)
        self.encoder = nn.Sequential(
            nn.Conv1d(1, 128, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.Conv1d(128, 256, kernel_size=5, padding=2),
            nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Conv1d(256, 128, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.Conv1d(128, 1, kernel_size=5, padding=2),
            nn.Tanh() 
        )

    def forward(self, x):
        latent = self.encoder(x)
        return self.decoder(latent)


save Model checkpoint Function

In [ ]:
def save_checkpoint(model, optimizer, epoch, loss, save_dir="/kaggle/working/"):
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)
    
    path = os.path.join(save_dir, f"rvc_anime_epoch_{epoch}.pth")
    torch.save({
        'epoch': epoch,
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'loss': loss
    }, path)
    print(f"Saved Model: {path}")


Training

In [ ]:
def train_system(dataset_folder):
    device = CONFIG['device']
    print(f"Starting Training on: {device}")
    
    # Load Data
    dataset = AnimeVoiceDataset(dataset_folder, CONFIG['sample_rate'], CONFIG['max_length'])
    if len(dataset) == 0: return
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=CONFIG['batch_size'], shuffle=True)

    # Init
    model = RVC_AnimeModel().to(device)
    optimizer = optim.AdamW(model.parameters(), lr=2e-4) # เพิ่ม LR นิดหน่อย
    metric_tool = AudioMetrics(CONFIG['sample_rate'])
    loss_history = []

    print(f" Dataset size: {len(dataset)} files")
    print("-" * 50)

    for epoch in range(1, CONFIG['epochs'] + 1):
        
        model.train()
        running_loss = 0.0
        
        # สร้าง Progress Bar สำหรับ Epoch นี้
        progress_bar = tqdm(dataloader, desc=f"Epoch {epoch}/{CONFIG['epochs']}", unit="batch")
        
        for batch_idx, real_voice in enumerate(progress_bar):
            real_voice = real_voice.to(device)
            
            # Forward
            fake_voice = model(real_voice)
            
            # Loss Calculation
            total_loss, mel_l, wave_l, gen_mel, tgt_mel = metric_tool.calc_loss(fake_voice, real_voice)
            
            # Backward
            optimizer.zero_grad()
            total_loss.backward()
            optimizer.step()
            
            running_loss += total_loss.item()

            # --- UPDATE TQDM BAR ---
            # แสดงค่า Loss แบบ Realtime ที่ท้ายแถบ
            progress_bar.set_postfix({
                "Total": f"{total_loss.item():.4f}",
                "Mel": f"{mel_l.item():.4f}",  # ดูอันนี้เป็นหลักว่าเสียงคล้ายไหม
                "Wave": f"{wave_l.item():.4f}" # ดูอันนี้ว่ามี noise เยอะไหม
            })

        # จบ Epoch
        avg_loss = running_loss / len(dataloader)
        loss_history.append(avg_loss)

        # Save & Evaluate
        if epoch % CONFIG['save_interval'] == 0:
            save_checkpoint(model, optimizer, epoch, avg_loss)
            plot_metrics(loss_history, fake_voice, real_voice, gen_mel, tgt_mel, epoch)
            # พิมพ์บอกหน่อยว่า Save แล้ว (ใช้วิธี write เพื่อไม่ให้กวน bar)
            tqdm.write(f"Checkpoint saved at Epoch {epoch} | Avg Loss: {avg_loss:.4f}")

    print("\nTraining Completed")


Run all Code

In [ ]:
if __name__ == "__main__":
    MY_DATASET_PATH = "/kaggle/input/mix-datasetssound/MixDatasets/" 
    if not os.path.exists(MY_DATASET_PATH):
        os.makedirs(MY_DATASET_PATH)
        print(f"Error")
    else:
        train_system(MY_DATASET_PATH)
